# Intervention Plan Effectiveness Drivers — Explanatory Model

---

## 1. Problem Framing

### Business Problem

Hearth Haven delivers structured intervention plans to each resident covering six
categories: Safety, Psychosocial, Education, Physical Health, Legal, and
Reintegration. Social workers design these plans during case conferences and update
them throughout the resident's time in care. A critical operational question is
whether the *composition* and *achievement* of these plans is actually associated
with measurable improvements in resident health and education outcomes.

This pipeline answers the question: **Which intervention plan categories and
services are most strongly associated with improvements in resident health scores
and education progress?**

The deployed output is an **Intervention Effectiveness panel** on the analytics
dashboard, showing which plan categories and achievement patterns are most
associated with positive resident outcomes — giving social workers and leadership
a data-grounded view of which interventions are working.

### Who Cares About This

- **Social workers** — need to know which types of interventions are most
  associated with the outcomes they're trying to achieve, so they can prioritize
  case conference planning.
- **Organization leadership** — needs to report to donors and partner agencies
  on which programs are associated with resident improvement.
- **Case supervisors** — need to identify whether plan achievement rates vary
  systematically across residents in ways that suggest case management gaps.

### Predictive vs. Explanatory

This pipeline uses an **explanatory approach**. The goal is to understand which
intervention characteristics are associated with better outcomes — not to predict
individual resident outcomes (which is covered by the reintegration and risk
pipelines).

OLS and Logit coefficients are the primary output. They answer: holding other
factors constant, how much is a one-unit change in this intervention characteristic
associated with better outcomes?

### Custom Feature Engineering

This pipeline requires custom feature engineering not covered by `prepare_residents()`,
because the question asks about the *composition* of intervention plans — information
that is aggregated differently than the standard resident pipeline. The data
preparation section builds a custom feature matrix from:

- `intervention_plans` — plan categories, services, achievement rates per category
- `health_wellbeing_records` — earliest and latest health scores per resident
- `education_records` — earliest and latest education progress per resident

### A Note on Causation

Even with significant coefficients, we cannot claim that specific intervention
types *cause* better outcomes. Residents receiving certain plan types may differ
systematically in ways not captured in the data (e.g., residents with Psychosocial
plans may have had more severe initial presentations). The coefficients describe
associations in the observational data.

---
## 2. Data Acquisition & Custom Feature Engineering

In [ ]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

sys.path.append(os.path.dirname(os.path.abspath('.')))
os.chdir('..')

from functions.fn_domain_prep import _get_data
from functions.fn_model_causal import (
    fit_causal_regression,
    fit_causal_classification,
    get_coefficients,
    check_assumptions,
    check_vif,
    refit_with_robust_se,
    run_stepwise_pvalue,
)

print("All imports successful.")

### 2.1 Load Raw Tables

We load three tables directly and build a custom feature matrix. This pipeline
does not use `prepare_residents()` because the unit of analysis and features are
different from the standard resident pipeline.

In [ ]:
interventions = _get_data('intervention_plans')
health        = _get_data('health_wellbeing_records')
education     = _get_data('education_records')
residents     = _get_data('residents')

print(f"intervention_plans:       {interventions.shape}")
print(f"health_wellbeing_records: {health.shape}")
print(f"education_records:        {education.shape}")
print(f"residents:                {residents.shape}")

# Parse dates
health['record_date']    = pd.to_datetime(health['record_date'],    errors='coerce')
education['record_date'] = pd.to_datetime(education['record_date'], errors='coerce')

# Numeric columns
health['general_health_score'] = pd.to_numeric(health['general_health_score'], errors='coerce')
education['progress_percent']  = pd.to_numeric(education['progress_percent'],  errors='coerce')

### 2.2 Engineer Outcome Features

For each resident we compute:
- **Health improvement:** latest health score − earliest health score
- **Education improvement:** latest progress % − earliest progress %
- **Composite outcome:** 1 if health improved > 0.2 points OR education improved
  > 5 percentage points

These thresholds represent clinically meaningful change — small noise fluctuations
in monthly assessments will not trigger the flag.

In [ ]:
# Health: earliest and latest scores per resident
health_sorted = health.sort_values('record_date')

health_first = (health_sorted.groupby('resident_id')['general_health_score']
                              .first().rename('health_score_first'))
health_last  = (health_sorted.groupby('resident_id')['general_health_score']
                              .last().rename('health_score_last'))
health_delta = (health_last - health_first).rename('health_improvement')

# Education: earliest and latest progress per resident
edu_sorted = education.sort_values('record_date')

edu_first = (edu_sorted.groupby('resident_id')['progress_percent']
                        .first().rename('progress_first'))
edu_last  = (edu_sorted.groupby('resident_id')['progress_percent']
                        .last().rename('progress_last'))
edu_delta = (edu_last - edu_first).rename('education_improvement')

# Join deltas
outcomes = pd.DataFrame({
    'health_improvement':    health_delta,
    'education_improvement': edu_delta,
}).reset_index()

# Composite target: meaningful improvement in either dimension
HEALTH_THRESHOLD    = 0.20  # health score units (1.0–5.0 scale)
EDUCATION_THRESHOLD = 5.0   # percentage points

outcomes['outcome_improved'] = (
    (outcomes['health_improvement']    > HEALTH_THRESHOLD) |
    (outcomes['education_improvement'] > EDUCATION_THRESHOLD)
).astype(int)

print(f"Residents with outcome data: {len(outcomes)}")
print(f"\nOutcome distribution:")
print(outcomes['outcome_improved'].value_counts())
print(f"\nPositive rate: {outcomes['outcome_improved'].mean():.1%}")
print(f"\nHealth improvement stats:")
print(outcomes['health_improvement'].describe().round(3))
print(f"\nEducation improvement stats:")
print(outcomes['education_improvement'].describe().round(1))

### 2.3 Engineer Intervention Features

For each resident we compute:
- **Plan counts per category** — how many plans in each of the 6 categories
- **Achievement rate per category** — % of plans in each category that are Achieved
- **Overall achievement rate** — % of all plans Achieved
- **Plan diversity** — number of distinct categories covered
- **Services provided** — binary flags for key service types

In [ ]:
PLAN_CATEGORIES = ['Safety', 'Psychosocial', 'Education',
                   'Physical Health', 'Legal', 'Reintegration']

# Plan counts per category
for cat in PLAN_CATEGORIES:
    col = f'plans_{cat.lower().replace(" ", "_")}'
    interventions[col] = (interventions['plan_category'] == cat).astype(int)

# Overall stats per resident
plan_stats = interventions.groupby('resident_id').agg(
    total_plans          =('plan_id',      'count'),
    plan_diversity       =('plan_category','nunique'),
    overall_achievement  =('status', lambda x: (x == 'Achieved').sum() / len(x)),
    open_plans           =('status', lambda x: (x == 'Open').sum()),
).reset_index()

# Plans per category
plan_counts = (interventions.groupby('resident_id')
               [[f'plans_{c.lower().replace(" ","_")}' for c in PLAN_CATEGORIES]]
               .sum()
               .reset_index())

# Achievement rate per category
def cat_achievement_rate(group, category):
    cat_plans = group[group['plan_category'] == category]
    if len(cat_plans) == 0:
        return 0.0
    return (cat_plans['status'] == 'Achieved').sum() / len(cat_plans)

for cat in PLAN_CATEGORIES:
    col = f'achieve_{cat.lower().replace(" ", "_")}'
    rates = (interventions.groupby('resident_id')
             .apply(lambda g, c=cat: cat_achievement_rate(g, c), include_groups=False)
             .reset_index(name=col))
    plan_stats = plan_stats.merge(rates, on='resident_id', how='left')

# Merge plan counts
plan_features = plan_stats.merge(plan_counts, on='resident_id', how='left')

print(f"Intervention features: {plan_features.shape}")
print(plan_features.head(3).to_string())

In [ ]:
# Join everything into one modeling DataFrame
df_model = (outcomes
            .merge(plan_features, on='resident_id', how='inner')
            .merge(residents[['resident_id', 'initial_risk_level',
                               'case_category', 'has_special_needs']],
                   on='resident_id', how='left'))

print(f"\nFinal modeling dataset: {df_model.shape}")
print(f"Residents with both outcomes and intervention data: {len(df_model)}")
print(f"\nMissing values:")
print(df_model.isnull().sum()[df_model.isnull().sum() > 0])

In [ ]:
# Fill any remaining nulls
df_model = df_model.fillna(0)

# Feature lists
INTERVENTION_NUMERIC = (
    ['total_plans', 'plan_diversity', 'overall_achievement', 'open_plans'] +
    [f'plans_{c.lower().replace(" ","_")}' for c in PLAN_CATEGORIES] +
    [f'achieve_{c.lower().replace(" ","_")}' for c in PLAN_CATEGORIES]
)

CONTROL_CATEGORICAL = ['initial_risk_level', 'case_category', 'has_special_needs']

ALL_FEATURES = INTERVENTION_NUMERIC + CONTROL_CATEGORICAL
TARGET = 'outcome_improved'

print(f"Intervention features: {len(INTERVENTION_NUMERIC)}")
print(f"Control variables:     {len(CONTROL_CATEGORICAL)}")

### 2.4 Exploratory Confirmation

Before modeling, we examine which intervention characteristics show the strongest
raw associations with the outcome.

In [ ]:
# Outcome rate by plan diversity
outcome_by_diversity = (df_model.groupby('plan_diversity')[TARGET]
                                 .agg(['mean', 'count'])
                                 .rename(columns={'mean': 'outcome_rate', 'count': 'n'}))
print("Outcome rate by plan diversity:")
print(outcome_by_diversity.round(3).to_string())

# Correlation of numeric intervention features with outcome
feature_df = df_model[INTERVENTION_NUMERIC]
corr = feature_df.corrwith(df_model[TARGET]).sort_values(key=abs, ascending=False)
print(f"\nTop intervention features by |correlation| with {TARGET}:")
print(corr.head(12).round(3).to_string())

In [ ]:
# Achievement rate by plan category — side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean achievement rate by category
cat_achieve = {c: df_model[f'achieve_{c.lower().replace(" ","_")}'].mean()
               for c in PLAN_CATEGORIES}
pd.Series(cat_achieve).sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Mean Achievement Rate by Plan Category')
axes[0].set_xlabel('Achievement Rate')

# Mean plan count by category
cat_counts = {c: df_model[f'plans_{c.lower().replace(" ","_")}'].mean()
              for c in PLAN_CATEGORIES}
pd.Series(cat_counts).sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Mean Plans per Resident by Category')
axes[1].set_xlabel('Mean Plan Count')

plt.tight_layout()
plt.show()

In [ ]:
# Outcome rate by initial risk level (control variable)
if 'initial_risk_level' in df_model.columns:
    risk_outcome = (df_model.groupby('initial_risk_level')[TARGET]
                             .agg(['mean','count'])
                             .rename(columns={'mean':'outcome_rate','count':'n'})
                             .sort_values('outcome_rate', ascending=False))
    print("Outcome rate by initial risk level:")
    print(risk_outcome.round(3).to_string())

---
## 3. Causal Model Specification

### 3.1 Train/Test Split

We hold out a test set. Feature selection and coefficient interpretation use
training data only.

In [ ]:
from functions.fn_prepare import split_data

X = df_model[ALL_FEATURES].copy()
y = df_model[TARGET].copy()

# Encode categorical controls
X = pd.get_dummies(X, columns=CONTROL_CATEGORICAL, drop_first=True, dtype=int)
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

X_train, X_test, y_train, y_test = split_data(X, y, stratify=True)

print(f"Train: {len(X_train)} rows  |  Test: {len(X_test)} rows")
print(f"Feature matrix: {X.shape[1]} columns")

### 3.2 Multicollinearity Check (VIF)

In [ ]:
print("Checking VIF...")
vif_df   = check_vif(X_train, threshold=10.0)
high_vif = vif_df[vif_df['VIF'] > 10]['feature'].tolist()            if 'VIF' in vif_df.columns else []

print(f"Features with VIF > 10: {high_vif if high_vif else 'None'}")
X_clean = X_train.drop(columns=high_vif, errors='ignore')
print(f"After cleanup: {X_clean.shape[1]} features remaining")

### 3.3 Fit Explanatory Logit Model

In [ ]:
# Classification target: fit_causal_classification (Logit)
results = fit_causal_classification(X_clean, y_train.astype(int))
print(results.summary())

### 3.4 Stepwise p-value Feature Selection

In [ ]:
results_final, kept_features = run_stepwise_pvalue(
    X_clean, y_train.astype(int),
    p_threshold=0.05,
    model_type='logistic',
)

print(f"\nFinal model: {len(kept_features)} significant features")
print(results_final.summary())

---
## 4. Evaluation & Interpretation

In [ ]:
coef_df = get_coefficients(results_final, model_type='logistic')

print("Significant coefficients (sorted by odds ratio):")
sig = coef_df[coef_df['p_value'] < 0.05].sort_values('odds_ratio', ascending=False)
if len(sig) > 0:
    print(sig[['feature','coefficient','odds_ratio','p_value','significant']]
          .to_string(index=False))
else:
    print("None at p < 0.05 — showing directional trends:")
    print(coef_df[['feature','coefficient','odds_ratio','p_value']]
          .sort_values('p_value').head(8).to_string(index=False))

os.makedirs('models', exist_ok=True)
coef_df.to_csv('models/intervention_effectiveness_coefficients.csv', index=False)
print("\nCoefficient table saved.")

In [ ]:
# Visualize
if len(sig) > 0:
    sig_plot = sig.set_index('feature')['coefficient'].sort_values()
    colors   = ['coral' if v < 0 else 'steelblue' for v in sig_plot]
    sig_plot.plot(kind='barh',
                  figsize=(10, max(4, len(sig_plot) * 0.45)),
                  color=colors)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title('Intervention Effectiveness — Significant Coefficients\n'
              '(Positive = associated with better health/education outcomes)', fontsize=12)
    plt.xlabel('Coefficient (log-odds)')
    plt.tight_layout()
    plt.show()

In [ ]:
print(f"Model fit statistics:")
print(f"  Pseudo R² (McFadden): {results_final.prsquared:.4f}")
print(f"  Log-Likelihood:       {results_final.llf:.4f}")
print(f"  AIC:                  {results_final.aic:.4f}")

### 4.3 Causal Interpretation

**How to read the coefficients:**

Each coefficient is in log-odds units. `exp(coefficient)` = the odds ratio: how
much the odds of outcome improvement multiply for a one-unit increase in that
feature, holding all else constant.

**Expected associations:**

1. **Achievement rate by category** — Higher achievement rates in Education and
   Physical Health plans are expected to show the strongest associations with
   measured outcome improvements, since these plans directly target the dimensions
   being measured (education progress and health scores).

2. **Plan diversity** — Residents with plans covering more categories may show
   better outcomes, reflecting more comprehensive case management. However, it
   could also reflect that more complex cases require more types of plans,
   potentially confounding the relationship.

3. **Psychosocial plan achievement** — Achieving Psychosocial goals (emotional
   regulation, trauma processing) is associated with improvements across all
   dimensions, consistent with research showing that mental health progress
   enables physical health and educational engagement.

**What we cannot claim causally:**

- The direction of causation is ambiguous. Residents who improve in health and
  education may be more able to engage with and achieve their intervention plans —
  the causal arrow may run in both directions.
- Residents receiving certain plan types may differ at intake in ways not fully
  captured by the `initial_risk_level` control variable.

**Actionable insight:**

The analysis provides social workers with a data-grounded basis for prioritizing
plan categories during case conferences. If Physical Health and Education plan
achievement rates show the strongest associations with measured outcomes, social
workers can use this to justify allocating more counseling time and resources
toward ensuring those specific plans are completed rather than merely opened.

### 4.4 Limitations

- **Small n:** With ~61 residents, coefficient estimates have wide confidence
  intervals. Direction is more reliable than magnitude.
- **Composite outcome simplification:** The binary outcome (improved vs. not)
  loses information about *how much* improvement occurred. A regression on the
  improvement delta would be more informative with more data.
- **Simultaneity bias:** Intervention plans and outcomes evolve together over
  time. A resident who improves may get different plans than one who doesn't,
  making it hard to distinguish whether interventions caused improvement or
  improvement caused different intervention patterns.

---
## 5. Deployment

This is an explanatory pipeline. Coefficient table served statically.

In [ ]:
summary = {
    'top_effective_interventions': (
        sig.nlargest(5, 'odds_ratio')[['feature','odds_ratio','p_value']]
        .to_dict('records') if len(sig) > 0 else []
    ),
    'model_pseudo_r2': round(results_final.prsquared, 4),
    'n_residents':     int(results_final.nobs),
    'model_version':   'intervention_effectiveness_v1',
}

import json
with open('models/intervention_effectiveness_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("Summary saved: models/intervention_effectiveness_summary.json")
print(json.dumps(summary, indent=2))

---
## 6. API Response Reference

```json
GET /api/analysis/intervention-effectiveness

{
  "top_effective_interventions": [
    {
      "feature": "string",
      "odds_ratio": "float",
      "p_value": "float",
      "interpretation": "string"
    }
  ],
  "model_pseudo_r2": "float",
  "n_residents": "int",
  "model_version": "intervention_effectiveness_v1",
  "generated_at": "ISO datetime"
}
```

**top_effective_interventions** — The intervention characteristics most strongly
associated with positive resident outcomes (better health and education scores),
sorted by odds ratio. Used to populate the Intervention Effectiveness panel on
the analytics dashboard.

**No `endpoints.py` or `server.py` changes needed.** The .NET backend reads
`models/intervention_effectiveness_summary.json` directly.

---
*Hearth Haven — IS 455 INTEX Pipeline*